# Notebook 1: Data Preprocessing and EDA

**Objective:** Transform the 600,000 records raw dataset from an "exercise-per-row" structure to a "routine-per-row" structure. This transformation handles the data aggregation challenge to ensure the correct input tensor dimensionality for the subsequent Keras DNN model.

*Execution context: This notebook should be executed locally.*

In [2]:
%pip install pandas numpy

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../') # allow imports from src folder

# Set pandas display options for easier analysis
pd.set_option('display.max_columns', None)

## 1. Load Raw Dataset

Load the `.csv` containing the exercise-level records.

In [6]:
import os
import sys
import pandas as pd
import numpy as np

# Determine the base path depending on our current working directory
base_path = '../' if os.path.basename(os.getcwd()) == 'notebooks' else './'
sys.path.append(base_path) # allow imports from src folder

pd.set_option('display.max_columns', None)

In [7]:
raw_data_path = os.path.join(base_path, 'data/raw/dataset.csv')
df_raw = pd.read_csv(raw_data_path)

# Display basic statistics and shape
print(f"Initial shape: {df_raw.shape}")
display(df_raw.head())

Initial shape: (605033, 16)


,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Knee-to-wall ankle dorsiflexion test,3.0,-180.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
1,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Banded ankle distractions,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
2,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Slant board calf stretch,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
3,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Seated Tibialis Raise,3.0,15.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
4,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,90/90 Hip Rotations,3.0,8.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00


In [8]:
# Check unique values per column
print("Unique elements per column:")
display(df_raw.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_raw.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_raw == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2598
description            2530
level                    47
goal                    250
equipment                 4
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          3213
sets                     18
reps                    205
intensity                11
created                2684
last_edit               623
dtype: int64

Total missing or '[]' values per column:


title                     0
description             819
level                  2936
goal                   2936
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [9]:
# Get an array of all unique exercise names
unique_exercises = df_raw['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 3213



<StringArray>
[   'Knee-to-wall ankle dorsiflexion test',
               'Banded ankle distractions',
                'Slant board calf stretch',
                   'Seated Tibialis Raise',
                     '90/90 Hip Rotations',
   'Pigeon Stretch with Thoracic Rotation',
                           'Hip Airplanes',
             'Quadruped T-Spine Rotations',
                     'Hanging Lat Stretch',
              'Banded Shoulder Dislocates',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 3213, dtype: str

In [10]:
# 1. Filter the dataset for Garage Gym
garage_gym_df = df_raw[df_raw['equipment'] == 'Garage Gym']

# 2. Extract unique exercise names from the filtered data
garage_exercises = garage_gym_df['exercise_name'].dropna().unique()

# Print the count and display the list
print(f"Total Garage Gym exercises: {len(garage_exercises)}\n")
display(garage_exercises)

Total Garage Gym exercises: 1128



<StringArray>
[ 'Knee-to-wall ankle dorsiflexion test',
             'Banded ankle distractions',
              'Slant board calf stretch',
                 'Seated Tibialis Raise',
                   '90/90 Hip Rotations',
 'Pigeon Stretch with Thoracic Rotation',
                         'Hip Airplanes',
           'Quadruped T-Spine Rotations',
                   'Hanging Lat Stretch',
            'Banded Shoulder Dislocates',
 ...
         'Bent Knee Standing Calf Raise',
                     '21 Curl (Barbell)',
                           'Waiter Curl',
          'Standing Leg Raise with Band',
                    'Leg Curl with Band',
               'Leg Extension with Band',
                             'Yates Row',
                               '1km Row',
                        'Band Push-down',
                            'Ghd Sit Up']
Length: 1128, dtype: str

In [11]:
# Create a new dataframe without 'Garage Gym' rows, leaving df_raw untouched
df_filtered = df_raw[df_raw['equipment'] != 'Garage Gym'].copy()

# Print both shapes to verify your original dataset is safe!
print(f"Original shape (df_raw): {df_raw.shape}")
print(f"Filtered shape (df_filtered): {df_filtered.shape}")

Original shape (df_raw): (605033, 16)
Filtered shape (df_filtered): (507947, 16)


In [12]:
# Check unique values per column
print("Unique elements per column:")
display(df_filtered.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_filtered.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_filtered == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2042
description            1990
level                    46
goal                    239
equipment                 3
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          2666
sets                     18
reps                    190
intensity                11
created                2107
last_edit               516
dtype: int64

Total missing or '[]' values per column:


title                     0
description             710
level                  2578
goal                   2578
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [13]:
# Get an array of all unique exercise names
unique_exercises = df_filtered['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 2666



<StringArray>
[                  'Bench Press (Barbell)',
                 'Bent Over Row (Barbell)',
                'Shoulder Press (Machine)',
                            'Lat Pulldown',
                'Tricep Extension (Cable)',
                 'Preacher Curl (Barbell)',
                         'Squat (Barbell)',
                      'Stiff Leg Deadlift',
                               'Leg Press',
                                'Leg Curl',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 2666, dtype: str

In [14]:
# Count the number of rows with negative reps values
num_negative_reps = (df_filtered['reps'] < 0).sum()

# Calculate the average reps value excluding the negative values
avg_reps_non_negative = df_filtered.loc[df_filtered['reps'] >= 0, 'reps'].mean()

print(f"Number of rows with negative reps: {num_negative_reps}")
print(f"Average reps (excluding negatives): {avg_reps_non_negative}")

Number of rows with negative reps: 22204
Average reps (excluding negatives): 10.792690785044767


In [43]:
# Print each equipment type and the number of rows that use it
print(df_filtered['equipment'].value_counts())

equipment
Full Gym         464904
At Home           29934
Dumbbell Only     13093
Name: count, dtype: int64


In [15]:
# Apply the filters
df_filtered = df_filtered[
    (df_filtered['equipment'] == "Full Gym")
].copy()
# Remove the specified columns
df_final = df_filtered.drop(columns=['description', 'created', 'last_edit'])

# Display the first few rows to verify they are gone
display(df_final.head())
# Display total rows
print(f"Total rows after filtering: {len(df_final)}")
#Solo equipment Full Gym, sin reps negativos, sin level vacíos o con '[]' como string.

,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity
325,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bench Press (Barbell),5.0,9.0,8.0
326,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bent Over Row (Barbell),5.0,9.0,8.0
327,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Shoulder Press (Machine),3.0,14.0,8.0
328,Lyle McDonald Routine (Strength/Hypertrophy Vers),"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Full Gym,12.0,90.0,1.0,1.0,6.0,Lat Pulldown,3.0,14.0,8.0
329,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Tricep Extension (Cable),2.0,5.0,8.0


Total rows after filtering: 464904


In [16]:
# Display 10 random rows
df_final.sample(n=min(10, len(df_final)))

,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity
357678,~The Formless Nosferatu~ (5 Days Weekly A&B Ro...,['Intermediate'],"['Bodybuilding', 'Athletics', 'Muscle & Sculpt...",Full Gym,12.0,80.0,3.0,4.0,5.0,Squat (Barbell),2.0,15.0,6.0
229955,Peace Pump,"['Intermediate', 'Advanced']","['Athletics', 'Bodybuilding', 'Powerbuilding']",Full Gym,12.0,70.0,5.0,1.0,8.0,AD Press (Smith Machine),1.0,10.0,8.0
203021,Villain Arc 3.0,"['Novice', 'Intermediate']","['Bodyweight Fitness', 'Powerbuilding']",Full Gym,12.0,60.0,8.0,3.0,5.0,Romanian Deadlift (Barbell),4.0,10.0,8.0
151181,Fullbody workout,"['Intermediate', 'Beginner', 'Novice']",['Bodybuilding'],Full Gym,12.0,10.0,6.0,2.0,5.0,Bench Press (Barbell),3.0,10.0,8.0
168205,Arnold (Winter-Spring 2025),['Intermediate'],"['Powerbuilding', 'Bodyweight Fitness']",Full Gym,16.0,100.0,7.0,5.0,6.0,Lying Bicep Curl,3.0,15.0,8.0
582065,SPLIT BUSTER WOLF,['Intermediate'],['Powerlifting'],Full Gym,4.0,90.0,3.0,3.0,8.0,Calf Raise (Leg Press),4.0,12.0,8.0
265001,Bro split#1,"['Novice', 'Intermediate']",['Bodybuilding'],Full Gym,12.0,70.0,12.0,5.0,6.0,Lat Pulldown,1.0,10.0,7.0
462320,Nicole's Super Awesome Workout Program,['Intermediate'],['Bodybuilding'],Full Gym,6.0,60.0,4.0,2.0,4.0,Seated Shoulder Press (Dumbbell),4.0,16.0,9.0
371935,Ivysaur Intermediate,['Intermediate'],"['Powerlifting', 'Bodybuilding']",Full Gym,12.0,90.0,4.0,2.0,6.0,Bench Press (Barbell),3.0,10.0,8.0
171051,JCKD Intellectual,"['Intermediate', 'Novice']","['Bodybuilding', 'Powerbuilding', 'Olympic Wei...",Full Gym,18.0,60.0,1.0,2.0,6.0,Leg Press,3.0,14.0,8.0


In [17]:
#check for null or negative values in sets, reps, intensity
import pandas as pd

# Create a summary table for the specific columns
summary_table = pd.DataFrame({
    'Null_Count': df_final[['sets', 'reps', 'intensity']].isnull().sum(),
    'Negative_Count': (df_final[['sets', 'reps', 'intensity']] < 0).sum()
})

# Display the table
display(summary_table)

,Null_Count,Negative_Count
sets,0,0
reps,0,20154
intensity,0,0


In [18]:
# Check what the negative values actually are to see if they are a code
print(df_final[df_final['reps'] < 0]['reps'].value_counts())

reps
-60.0      4153
-120.0     2165
-300.0     1427
-900.0     1349
-1800.0    1260
           ... 
-1620.0       1
-1740.0       1
-1980.0       1
-2040.0       1
-930.0        1
Name: count, Length: 122, dtype: int64


In [19]:
#Calculate Systemic Load per exercise
#S = sets * reps * intensity
import numpy as np

# Calculate systemic load: Sets * Reps * Intensity (but only if Reps >= 0. Otherwise, leave as NaN)
df_final['systemic_load'] = np.where(
    df_final['reps'] >= 0, 
    df_final['sets'] * df_final['reps'] * df_final['intensity'], 
    np.nan
)

# Display the result to verify
display(df_final[['sets', 'reps', 'systemic_load']].head())

,sets,reps,systemic_load
325,5.0,9.0,360.0
326,5.0,9.0,360.0
327,3.0,14.0,336.0
328,3.0,14.0,336.0
329,2.0,5.0,80.0


## 2. Data Aggregation (Exercise to Routine Level)

We aggregate the dataset using custom mapping rules (e.g. summing total weight/reps for systemic load, averaging intensities). This directly builds the input vectors required for the Keras `input_shape`.

**Academic Justification:** Deep Learning models require a uniform, structured input vector for each instance. Aggregating the sequential/tabular exercise steps into a single "routine" feature vector ensures the DNN receives the global physiological snapshot of the workout (volume, load, etc.) rather than disjointed exercises.

In [20]:
#Check if any routine has ALL exercises with missing or empty levels
import pandas as pd

def check_missing_levels_by_routine(df):
    """
    Groups the dataframe by routine ('title') and evaluates if ALL rows 
    within each routine have a missing or empty ('[]') level.
    
    Returns a dataframe with each routine and a True/False flag.
    """
    # 1. Create a boolean mask: True if level is NaN or the string representation is '[]'
    is_missing = df['level'].isna() | (df['level'].astype(str) == '[]')
    
    # 2. Attach this temporary check to the dataframe
    df_temp = df.assign(is_level_missing=is_missing)
    
    # 3. Group by title and use .all() 
    # .all() returns True ONLY if every single row in that group is True
    agg_df = df_temp.groupby('title')['is_level_missing'].all().reset_index()
    
    # Rename the column so it's easy to read
    agg_df = agg_df.rename(columns={'is_level_missing': 'all_levels_missing'})
    
    return agg_df

# --- 1. RUN THE CHECK ---
df_missing_check = check_missing_levels_by_routine(df_final)

# See the results
display(df_missing_check.head())

#See exactly WHICH routines have completely missing levels:
completely_missing_routines = df_missing_check[df_missing_check['all_levels_missing'] == True]

print(f"\nThere are {len(completely_missing_routines)} routines where EVERY exercise is missing a level.")
if len(completely_missing_routines) > 0:
    display(completely_missing_routines)

,title,all_levels_missing
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,False
1,(NOT MY PROGRAM)SHJ Jotaro,False
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,False
3,10 week deadlift focus,False
4,1000 lbs Club,False



There are 2 routines where EVERY exercise is missing a level.


,title,all_levels_missing
986,Mania (Upper/Lower),True
1727,Viking strong,True


In [21]:
def display_routine_rows(df, routine_title):
    """
    Filters the dataframe to show only the rows that belong to a specific routine.
    """
    # Filter the dataset where the 'title' exactly matches the requested routine
    routine_df = df[df['title'] == routine_title]
    
    print(f"Showing {len(routine_df)} rows for routine: '{routine_title}'")
    
    # Display the filtered dataframe
    display(routine_df)
    
    # (Optional) Return it in case you want to save it to a variable
    return routine_df


In [22]:
#Check the routine program 
my_routine_data = display_routine_rows(df_final, "Mania (Upper/Lower)")

Showing 288 rows for routine: 'Mania (Upper/Lower)'


,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,systemic_load
513536,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Squat (Barbell),3.0,15.0,7.0,315.0
513537,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Seated Row (Cable),2.0,12.0,7.0,168.0
513538,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Hamstring Curl,2.0,12.0,7.0,168.0
513539,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Standing Calf Raise,3.0,3.0,8.0,72.0
513540,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Leg Extension,2.0,12.0,8.0,192.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513819,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Chest Supported Row (Dumbbell),3.0,5.0,7.0,105.0
513820,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Tricep Pushdown (Cable),2.0,5.0,8.0,80.0
513821,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Incline Curl (Dumbbell),2.0,11.0,8.0,176.0
513822,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Lateral Raise (Dumbbell),2.0,9.0,7.0,126.0


In [23]:
# --- 2. REMOVE THE BAD ROUTINES ---
# Extract the titles of the bad routines
bad_titles = completely_missing_routines['title']
# Filter df_final to keep only rows where the title is NOT (~) in the bad_titles list
df_final = df_final[~df_final['title'].isin(bad_titles)]
print(f"Successfully removed those {len(bad_titles)} routines from df_final")

Successfully removed those 2 routines from df_final


In [30]:
# --- GET WEEKLY VOLUME PER ROUTINE ---
def get_avg_weekly_volume(df):
    """
    Calculates the average weekly volume for each routine, 
    relying on the existing 'week' column.
    """
    # --- STEP 1: Get the volume per week ---
    # We sum() the volume to get the total volume of that routine for each week.
    weekly_totals = df.groupby(['title', 'week'])['sets'].sum().round(2).reset_index()
    
    # --- STEP 2: Get the average of those weeks ---
    # Now we group just by title, and get the mean() of those weekly totals
    avg_weekly_volume = weekly_totals.groupby('title')['sets'].mean().round(2).reset_index()
    
    # Rename the column so it's clear
    avg_weekly_volume = avg_weekly_volume.rename(columns={'sets': 'avg_weekly_volume'})
    
    return avg_weekly_volume


# --- How to use it ---
df_avg_weekly_volume = get_avg_weekly_volume(df_final)
#display 30 examples
display(df_avg_weekly_volume.head(10))

,title,avg_weekly_volume
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,54.00
1,(NOT MY PROGRAM)SHJ Jotaro,96.00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,98.00
3,10 week deadlift focus,112.30
4,1000 lbs Club,108.00
5,10x3 Powerbuilding,62.67
6,12 WEEK FAT DESTROYER.,92.00
7,12 Week Amped Up,112.00
8,12 Week Women’s Bikini Body Program,54.00
9,12 Weeks Bikini Comp Ready,91.33


In [24]:
#-- GET DAILY VOLUME PER ROUTINE ---
def get_avg_daily_volume(df):
    """
    Calculates the average daily volume per routine, 
    where 'volume' is defined as the total number of sets.
    """
    # --- STEP 1: Get total sets per day ---
    # Group by title, week, and day, then sum() the sets to get the total sets done in that single session
    daily_totals = df.groupby(['title', 'week', 'day'])['sets'].sum().round(2).reset_index()
    
    # --- STEP 2: Get the average of those days ---
    # Now group just by title, and get the mean() of those daily session totals
    avg_daily_volume = daily_totals.groupby('title')['sets'].mean().round(2).reset_index()
    
    # Rename the column so it's clear we are measuring by sets
    avg_daily_volume = avg_daily_volume.rename(columns={'sets': 'avg_daily_volume_sets'})
    
    return avg_daily_volume

# --- How to use it ---
df_avg_daily_volume = get_avg_daily_volume(df_final)

display(df_avg_daily_volume.head(10))

,title,avg_daily_volume_sets
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,7.71
1,(NOT MY PROGRAM)SHJ Jotaro,24.00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,19.60
3,10 week deadlift focus,23.40
4,1000 lbs Club,18.00
5,10x3 Powerbuilding,15.67
6,12 WEEK FAT DESTROYER.,23.00
7,12 Week Amped Up,18.67
8,12 Week Women’s Bikini Body Program,13.50
9,12 Weeks Bikini Comp Ready,13.05


In [25]:
# -- GET WEEKLY SYSTEMIC LOAD ---
def get_avg_weekly_systemic_load(df):
    """
    Calculates the average weekly systemic load for each routine.
    Assumes you have a column named 'systemic_load'.
    """
    # --- STEP 1: Get total systemic load per week ---
    # Group by title and week, then sum() to get the total load endured in that single week
    weekly_totals = df.groupby(['title', 'week'])['systemic_load'].sum().reset_index()
    
    # --- STEP 2: Get the average of those weeks ---
    # Group by title and get the mean() of those weekly totals
    avg_weekly_load = weekly_totals.groupby('title')['systemic_load'].mean().round(2).reset_index()
    
    # Rename the column to exactly what you requested
    avg_weekly_load = avg_weekly_load.rename(columns={'systemic_load': 'avg_systemic_load_per_week'})
    
    return avg_weekly_load

# --- How to use it ---
df_avg_systemic_load = get_avg_weekly_systemic_load(df_final)

display(df_avg_systemic_load.head(10))

,title,avg_systemic_load_per_week
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,4378.50
1,(NOT MY PROGRAM)SHJ Jotaro,5390.50
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,7805.33
3,10 week deadlift focus,9705.80
4,1000 lbs Club,8355.17
5,10x3 Powerbuilding,6173.08
6,12 WEEK FAT DESTROYER.,8555.58
7,12 Week Amped Up,6615.67
8,12 Week Women’s Bikini Body Program,2748.00
9,12 Weeks Bikini Comp Ready,7081.83


In [26]:
def get_routine_intensity_stats(df):
    """
    Calculates both the average intensity and the maximum intensity 
    for the entire routine.
    """
    # Group by routine title and calculate mean and max simultaneously
    intensity_stats = df.groupby('title').agg(
        avg_intensity=('intensity', 'mean'),
        max_intensity=('intensity', 'max')
    ).round(2).reset_index()
    
    return intensity_stats

# --- How to use it ---
df_intensity = get_routine_intensity_stats(df_final)

display(df_intensity.head(10))

,title,avg_intensity,max_intensity
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,8.29,9.0
1,(NOT MY PROGRAM)SHJ Jotaro,7.10,8.0
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,8.35,10.0
3,10 week deadlift focus,7.37,9.0
4,1000 lbs Club,7.95,9.0
5,10x3 Powerbuilding,7.53,9.0
6,12 WEEK FAT DESTROYER.,7.89,10.0
7,12 Week Amped Up,8.86,10.0
8,12 Week Women’s Bikini Body Program,8.18,9.0
9,12 Weeks Bikini Comp Ready,7.66,9.0


In [27]:
# Calculate intensity variance per week and total per routine
import pandas as pd

def get_intensity_variance(df):
    """
    Calculates the total intensity variance for the entire routine, 
    and the average weekly intensity variance.
    """
    # --- 1. Total Variance per Routine ---
    # How much does intensity vary across the ENTIRE routine (all weeks/days combined)
    total_var = df.groupby('title')['intensity'].var()
    
    # --- 2. Variance per Week ---
    # First calculate the variance inside each specific week
    weekly_var = df.groupby(['title', 'week'])['intensity'].var()
    # Then take the average of those weekly variances to get a single number per routine
    avg_weekly_var = weekly_var.groupby('title').mean()
    
    # --- Combine into one table ---
    variance_df = pd.DataFrame({
        'intensity_variance_total': total_var,
        'intensity_variance_avg_weekly': avg_weekly_var
    }).round(2).reset_index()
    
    # Fill NaN values with 0 (Happens if a week only had 1 exercise)
    variance_df = variance_df.fillna(0)
    
    return variance_df

# --- How to use it ---
df_intensity_variance = get_intensity_variance(df_final)

display(df_intensity_variance.head())

,title,intensity_variance_total,intensity_variance_avg_weekly
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,0.61,0.61
1,(NOT MY PROGRAM)SHJ Jotaro,0.09,0.09
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,2.16,0.85
3,10 week deadlift focus,2.48,0.64
4,1000 lbs Club,1.12,0.55


In [37]:
def get_total_weeks_per_routine(df):
    """
    Calculates the total number of unique weeks for each routine.
    """
    # Group by routine title and count the number of UNIQUE weeks
    total_weeks = df.groupby('title')['week'].nunique().reset_index()
    
    # Rename column so it's clear
    total_weeks = total_weeks.rename(columns={'week': 'total_weeks'})
    
    return total_weeks

# --- How to use it ---
df_total_weeks = get_total_weeks_per_routine(df_final)

display(df_total_weeks.head())

,title,total_weeks
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,12
1,(NOT MY PROGRAM)SHJ Jotaro,8
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,6
3,10 week deadlift focus,10
4,1000 lbs Club,12


In [50]:
def get_avg_days_per_week(df):
    """
    Calculates the average number of workout days per week for each routine.
    Assumes you have a column named 'day' that identifies the day of the session.
    """
    # --- STEP 1: Count unique days per week ---
    # Group by routine and week, then use .nunique() to count how many distinct days exist in that week
    weekly_days = df.groupby(['title', 'week'])['day'].nunique().reset_index()
    
    # --- STEP 2: Average those weekly counts ---
    # Group by just the routine title, and calculate the mean() of those weekly day counts
    avg_days_per_week = weekly_days.groupby('title')['day'].mean().reset_index().round(2)
    
    # Rename the column so it's clear what the metric is
    avg_days_per_week = avg_days_per_week.rename(columns={'day': 'avg_days_per_week'})
    
    return avg_days_per_week

# --- How to use it ---
df_avg_days = get_avg_days_per_week(df_final)


display(df_avg_days.head())

,title,avg_days_per_week
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,7.0
1,(NOT MY PROGRAM)SHJ Jotaro,4.0
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,5.0
3,10 week deadlift focus,4.8
4,1000 lbs Club,6.0


In [47]:
def get_avg_exercises_per_day(df, exercise_column='exercise_name'):
    """
    Calculates the average number of unique exercises performed per day for each routine.
    Make sure to change 'exercise_column' to the actual name of your column!
    """
    # --- STEP 1: Count unique exercises per day ---
    # Group by routine, week, and day. 
    # Use .nunique() on the exercise column to count how many distinct exercises happened in that session.
    daily_exercise_count = df.groupby(['title', 'week', 'day'])[exercise_column].nunique().reset_index()
    
    # --- STEP 2: Average those daily counts ---
    # Group by just the routine title, and calculate the mean() of those daily counts
    avg_exercises = daily_exercise_count.groupby('title')[exercise_column].mean().reset_index()
    
    # Rename the column so it's clear in your final table
    avg_exercises = avg_exercises.rename(columns={exercise_column: 'avg_exercises_per_day'})
    
    return avg_exercises

# --- How to use it ---
# Change 'exercise' to whatever your actual column name is!
df_avg_exercises = get_avg_exercises_per_day(df_final, exercise_column='exercise_name')

display(df_avg_exercises.head())

,title,avg_exercises_per_day
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,4.571429
1,(NOT MY PROGRAM)SHJ Jotaro,6.000000
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,7.100000
3,10 week deadlift focus,7.416667
4,1000 lbs Club,4.500000


In [53]:
# -- Merge all results into one final table for analysis --
import pandas as pd

def compile_routine_summary(df):
    """
    Calls all of our modular calculation functions and merges 
    the results into a single final dataset.
    """
    # 1. Grab the 'level' for each routine
    # Since level doesn't change within a routine, we just take the first one we see
    df_level = df.groupby('title')[['level']].first().reset_index()
    
    # 2. Calculate all our metrics using the functions we defined earlier
    df_vol_weekly = get_avg_weekly_volume(df)
    df_vol_daily = get_avg_daily_volume(df)
    df_sys_load = get_avg_weekly_systemic_load(df)
    df_intensity = get_routine_intensity_stats(df)
    df_variance = get_intensity_variance(df)
    df_total_weeks = get_total_weeks_per_routine(df)
    df_avg_days = get_avg_days_per_week(df)
    df_avg_exercises = get_avg_exercises_per_day(df, exercise_column='exercise_name')
    
    
    # 3. Merge them all together
    # We start with our df_level as the base table
    final_df = df_level
    
    # Put all our calculated tables into a list
    dataframes_to_merge = [
        df_vol_weekly, 
        df_vol_daily, 
        df_sys_load, 
        df_intensity, 
        df_variance,
        df_total_weeks,
        df_avg_days,
        df_avg_exercises
    ]
    
    # Loop through the list and attach each one to the final_df
    for data in dataframes_to_merge:
        final_df = pd.merge(final_df, data, on='title', how='left')
        
    # Optional: Fill any NaN variances with 0 and round everything
    final_df = final_df.fillna(0).round(2)
        
    return final_df

# --- How to use it ---
# Make sure all your functions from earlier are run in your notebook, then just call:
df_final_aggregated = compile_routine_summary(df_final)

display(df_final_aggregated.head(30))
print(f"There are {len(df_final_aggregated)} routines in the final dataset.")

,title,level,avg_weekly_volume,avg_daily_volume_sets,avg_systemic_load_per_week,avg_intensity,max_intensity,intensity_variance_total,intensity_variance_avg_weekly,total_weeks,avg_days_per_week,avg_exercises_per_day
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,['Intermediate'],54.00,7.71,4378.50,8.29,9.0,0.61,0.61,12,7.0,4.57
1,(NOT MY PROGRAM)SHJ Jotaro,"['Advanced', 'Intermediate']",96.00,24.00,5390.50,7.10,8.0,0.09,0.09,8,4.0,6.00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,"['Beginner', 'Novice', 'Intermediate']",98.00,19.60,7805.33,8.35,10.0,2.16,0.85,6,5.0,7.10
3,10 week deadlift focus,"['Intermediate', 'Advanced']",112.30,23.40,9705.80,7.37,9.0,2.48,0.64,10,4.8,7.42
4,1000 lbs Club,['Intermediate'],108.00,18.00,8355.17,7.95,9.0,1.12,0.55,12,6.0,4.50
5,10x3 Powerbuilding,['Intermediate'],62.67,15.67,6173.08,7.53,9.0,1.04,0.25,12,4.0,5.25
6,12 WEEK FAT DESTROYER.,"['Beginner', 'Novice']",92.00,23.00,8555.58,7.89,10.0,0.94,0.51,12,4.0,6.00
7,12 Week Amped Up,"['Novice', 'Beginner']",112.00,18.67,6615.67,8.86,10.0,1.16,0.78,12,6.0,6.25
8,12 Week Women’s Bikini Body Program,"['Beginner', 'Novice']",54.00,13.50,2748.00,8.18,9.0,0.76,0.63,12,4.0,5.50
9,12 Weeks Bikini Comp Ready,['Intermediate'],91.33,13.05,7081.83,7.66,9.0,0.40,0.24,12,7.0,5.26


There are 1850 routines in the final dataset.


In [55]:
# Count the number of appearances of each exact combination of levels
level_frequencies = df_final_aggregated['level'].astype(str).value_counts()

# Show as a percentage to make it easier to analyze
level_percentages = df_final_aggregated['level'].astype(str).value_counts(normalize=True) * 100
print("\n--- Percentage of each combination (%) ---")
display(level_percentages.round(2))


--- Percentage of each combination (%) ---


level
['Intermediate']                                      23.14
['Intermediate', 'Advanced']                          12.16
['Novice', 'Intermediate']                            10.81
['Novice']                                             8.00
['Beginner']                                           6.86
['Beginner', 'Novice', 'Intermediate', 'Advanced']     4.86
['Beginner', 'Novice']                                 4.54
['Advanced']                                           4.43
['Beginner', 'Novice', 'Intermediate']                 3.84
['Intermediate', 'Novice']                             3.41
['Novice', 'Intermediate', 'Advanced']                 2.49
['Novice', 'Beginner']                                 2.16
['Advanced', 'Intermediate']                           2.05
['Intermediate', 'Advanced', 'Novice']                 1.57
['Novice', 'Intermediate', 'Beginner']                 1.19
['Intermediate', 'Novice', 'Advanced']                 1.14
['Beginner', 'Intermediate']      

In [61]:
import ast

def extract_entry_barrier(level_data):
    """
    Reads the list of levels and returns the LOWEST difficulty found.
    (This defines the routine by its minimum barrier to entry).
    """
    if isinstance(level_data, str):
        try:
            levels = ast.literal_eval(level_data)
        except:
            levels = [level_data]
    else:
        levels = level_data
        
    # Check for the LOWEST difficulty first!
    if 'Beginner' in levels:
        return 'Beginner'
    elif 'Novice' in levels:
        return 'Beginner'
    elif 'Intermediate' in levels:
        return 'Intermediate'
    elif 'Advanced' in levels:
        return 'Advanced'
    else:
        return 'Unknown'

# Apply the new logic
df_final_aggregated['target_level'] = df_final_aggregated['level'].apply(extract_entry_barrier)

# Let's see how this shifts the distribution!
clean_percentages = df_final_aggregated['target_level'].value_counts(normalize=True) * 100
print("--- 'Entry-Barrier' Target Distribution (%) ---")
display(clean_percentages.round(2))

--- 'Entry-Barrier' Target Distribution (%) ---


target_level
Beginner        58.00
Intermediate    37.35
Advanced         4.43
Unknown          0.22
Name: proportion, dtype: float64

In [64]:
# Drop the 4 'Unknown' routines
df_final_aggregated = df_final_aggregated[df_final_aggregated['target_level'] != 'Unknown']
df_final_aggregated = df_final_aggregated.drop(columns=['level'])

print(f"Dataset is now 100% clean with {len(df_final_aggregated)} routines!")

Dataset is now 100% clean with 1846 routines!


In [65]:
#Show the final table with all our calculated metrics and the new target variable
df_final_aggregated

,title,avg_weekly_volume,avg_daily_volume_sets,avg_systemic_load_per_week,avg_intensity,max_intensity,intensity_variance_total,intensity_variance_avg_weekly,total_weeks,avg_days_per_week,avg_exercises_per_day,target_level
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,54.00,7.71,4378.50,8.29,9.0,0.61,0.61,12,7.00,4.57,Intermediate
1,(NOT MY PROGRAM)SHJ Jotaro,96.00,24.00,5390.50,7.10,8.0,0.09,0.09,8,4.00,6.00,Intermediate
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,98.00,19.60,7805.33,8.35,10.0,2.16,0.85,6,5.00,7.10,Beginner
3,10 week deadlift focus,112.30,23.40,9705.80,7.37,9.0,2.48,0.64,10,4.80,7.42,Intermediate
4,1000 lbs Club,108.00,18.00,8355.17,7.95,9.0,1.12,0.55,12,6.00,4.50,Intermediate
...,...,...,...,...,...,...,...,...,...,...,...,...
1845,자기화 프로그램,145.00,24.17,12082.80,7.60,8.0,0.64,0.66,5,6.00,6.83,Beginner
1846,찬중,62.00,20.67,5502.33,8.44,9.0,0.25,0.25,3,3.00,6.00,Intermediate
1847,𝕮𝖍𝖎𝖒𝖊𝖗𝖆,22.00,5.50,1468.67,7.92,8.0,0.08,0.08,12,4.00,5.50,Beginner
1848,"🔥 ""Upper Body Dominance: 3-Day Growth System"" 🔥",32.00,10.67,1996.67,6.75,8.0,0.86,0.83,6,3.00,5.33,Beginner


## 3. Save Processed Dataset

Save the aggregated DataFrame to a CSV so it can be uploaded to Google Colab for Phase 2.

In [ ]:
df_final_aggregated.to_csv("../data/routines_model_ready.csv", index=False)
print("Saved successfully in the /data folder! Ready for Week 1 of modeling.")

OSError: Cannot save file into a non-existent directory: 'data'